In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
import os
import logging
from cryptography.fernet import Fernet



try:
    fernet_cipher = Fernet("HX3wjH63lEdvvISj_9BtdTLQdbpLodlh-fks3cnWYGY=".encode())
except Exception as e:
    # logger.error(f"Invalid ENCRYPTION_KEY format. Generating fallback. Error: {e}")
    fernet_cipher = Fernet(Fernet.generate_key())


def encrypt_token(token: str) -> str:
    """Encrypt a string token at rest."""
    if not token:
        return ""
    return fernet_cipher.encrypt(token.encode()).decode()


def decrypt_token(encrypted_token: str) -> str:
    """Decrypt an encrypted token."""
    if not encrypted_token:
        return ""
    try:
        return fernet_cipher.decrypt(encrypted_token.encode()).decode()
    except Exception as e:
        # logger.error(f"Failed to decrypt token: {e}")
        raise ValueError("Decryption failed. Invalid key or corrupted payload.")


In [3]:
def decrypt_token(encrypted_token: str) -> str:
    """Decrypt an encrypted token."""
    if not encrypted_token:
        return ""
    try:
        return fernet_cipher.decrypt(encrypted_token.encode()).decode()
    except Exception as e:
        # logger.error(f"Failed to decrypt token: {e}")
        raise ValueError("Decryption failed. Invalid key or corrupted payload.")

In [4]:
encrypted_token = os.getenv("GITHUB_ACCESS_TOKEN")
access_token = decrypt_token(encrypted_token)

headers={
    "Authorization": f"token {access_token}",
    "Accept": "application/vnd.github.v3+json",
    "User-Agent": "AI-Developer-Story-Platform",
}

In [5]:
import httpx

In [6]:
from typing import List, Dict, Any
client = httpx.AsyncClient(headers=headers, timeout=15.0)
async def list_repositories() -> List[Dict[str, Any]]:
    """List repositories for the authenticated user."""
    repos = []
    page = 1
    while True:
    # Fetch both public and private repos owned by user or where they contribute
        r = await client.get(
                "https://api.github.com/user/repos",
                params={"per_page": 100, "page": page, "sort": "pushed"},
            )
        r.raise_for_status()
        data = r.json()
        if not data:
                break
        repos.extend(data)
        if len(data) < 100:
            break
        page += 1
    return repos

In [7]:
async def get_repo_details(owner: str, repo: str) -> Dict[str, Any]:
        """Get repository metadata."""
        r = await client.get(f"https://api.github.com/repos/{owner}/{repo}")
        r.raise_for_status()
        return r.json()

In [8]:
await get_repo_details("Akshat3422", "SalesAutoAgent")

{'id': 1201458564,
 'node_id': 'R_kgDOR5zNhA',
 'name': 'SalesAutoAgent',
 'full_name': 'Akshat3422/SalesAutoAgent',
 'private': False,
 'owner': {'login': 'Akshat3422',
  'id': 202176233,
  'node_id': 'U_kgDODAz26Q',
  'avatar_url': 'https://avatars.githubusercontent.com/u/202176233?v=4',
  'gravatar_id': '',
  'url': 'https://api.github.com/users/Akshat3422',
  'html_url': 'https://github.com/Akshat3422',
  'followers_url': 'https://api.github.com/users/Akshat3422/followers',
  'following_url': 'https://api.github.com/users/Akshat3422/following{/other_user}',
  'gists_url': 'https://api.github.com/users/Akshat3422/gists{/gist_id}',
  'starred_url': 'https://api.github.com/users/Akshat3422/starred{/owner}{/repo}',
  'subscriptions_url': 'https://api.github.com/users/Akshat3422/subscriptions',
  'organizations_url': 'https://api.github.com/users/Akshat3422/orgs',
  'repos_url': 'https://api.github.com/users/Akshat3422/repos',
  'events_url': 'https://api.github.com/users/Akshat3422/eve

In [9]:
import base64
from typing import Optional
async def get_readme( owner: str, repo: str) -> Optional[str]:
        """Fetch and decode repository README."""
        try:
            r = await client.get(
                f"https://api.github.com/repos/{owner}/{repo}/readme"
            )
            if r.status_code == 404:
                return None
            r.raise_for_status()
            data = r.json()
            content_b64 = data.get("content", "")
            # Remove newlines before decoding
            decoded = base64.b64decode(content_b64.replace("\n", "")).decode(
                "utf-8", errors="ignore"
            )
            return decoded
        except Exception as e:
            # logger.warning(f"Error fetching README for {owner}/{repo}: {e}")
            return None

In [10]:
result=await get_readme("Akshat3422", "SalesAutoAgent")

In [11]:
from pathlib import Path
import os

IGNORE_DIRS = {
    "node_modules",
    "venv",
    ".venv",
    "__pycache__",
    ".git",
    "dist",
    "build",
    ".next",
    "coverage",
    ".idea",
    ".vscode",
}

IGNORE_FILES = {
    ".DS_Store",
    "package-lock.json",
    "yarn.lock",
    "pnpm-lock.yaml",
    ".env",
}

VALID_CODE_EXTENSIONS = {
    ".py",
    ".js",
    ".ts",
    ".tsx",
    ".jsx",
    ".java",
    ".go",
    ".rs",
    ".cpp",
    ".c",
    ".cs",
    ".php",
    ".rb",
    ".swift",
    ".kt",
    ".sql",
    ".html",
    ".css",
    ".scss",
    ".json",
    ".yaml",
    ".yml",
    ".md",
}

IMPORTANT_FILES = {
    "README.md",
    "requirements.txt",
    "package.json",
    "pyproject.toml",
    "Dockerfile",
    "docker-compose.yml",
}


def is_important_file(file_path: str) -> bool:
    path = Path(file_path)

    # Ignore directories
    if any(part in IGNORE_DIRS for part in path.parts):
        return False

    filename = path.name
    ext = path.suffix.lower()

    # Ignore specific files
    if filename in IGNORE_FILES:
        return False

    # Prioritize important files
    if filename in IMPORTANT_FILES:
        return True

    # Allow valid source code files
    if ext in VALID_CODE_EXTENSIONS:
        return True

    return False

In [12]:
async def repo_structure(owner:str,repo:str):
    """Fetch repository structure and file contents."""
    try:
        result=await client.get(f"https://api.github.com/repos/{owner}/{repo}/git/trees/main?recursive=1")
        result.raise_for_status()
        repo_data= result.json()
        file_paths=[item['path'] for item in repo_data['tree'] if item['type']=='blob']
        final_files = [path for path in file_paths if is_important_file(path)]
        return final_files
        
    except Exception as e:
        # logger.warning(f"Error fetching repository structure for {owner}/{repo}: {e}")
        return None
    

In [13]:
repo_data=await repo_structure("Akshat3422", "SalesAutoAgent")

In [14]:
repo_data

['API_RESPONSE_SHEET.json',
 'DEBUG_CHECKLIST.md',
 'FRONTEND_SETUP.md',
 'README.md',
 'demo.py',
 'email_service.py',
 'frontend/README.md',
 'frontend/eslint.config.js',
 'frontend/index.html',
 'frontend/package.json',
 'frontend/src/App.tsx',
 'frontend/src/api/client.ts',
 'frontend/src/components/AIAnalysisTab.tsx',
 'frontend/src/components/CampaignModal.tsx',
 'frontend/src/components/CampaignsTab.tsx',
 'frontend/src/components/CompaniesTab.tsx',
 'frontend/src/components/ContactsTab.tsx',
 'frontend/src/components/Header.tsx',
 'frontend/src/components/LoginPage.tsx',
 'frontend/src/components/LogsTab.tsx',
 'frontend/src/components/MainPanel.tsx',
 'frontend/src/components/MetricsPanel.tsx',
 'frontend/src/components/OutreachTab.tsx',
 'frontend/src/components/WorkflowSidebar.tsx',
 'frontend/src/hooks/useApi.ts',
 'frontend/src/index.css',
 'frontend/src/main.tsx',
 'frontend/src/store/authStore.ts',
 'frontend/src/store/campaignStore.ts',
 'frontend/src/types/index.ts',
 

In [15]:
async def get_file_content(owner: str, repo: str, path: str) -> str:
        """Fetch and decode the raw content of a specific file."""
        try:
            r = await client.get(
                f"https://api.github.com/repos/{owner}/{repo}/contents/{path}"
            )
            if r.status_code == 404:
                return ""
            r.raise_for_status()
            data = r.json()
            if isinstance(data, list):
                return ""
            content_b64 = data.get("content", "")
            return base64.b64decode(content_b64.replace("\n", "")).decode(
                "utf-8", errors="ignore"
            )
        except Exception as e:
            # logger.warning(f"Error reading file {path} from {owner}/{repo}: {e}")
            return ""

In [16]:
import asyncio

SEM = asyncio.Semaphore(10)

async def safe_fetch(file):
    async with SEM:
        try:
            return await get_file_content(
                "Akshat3422",
                "SalesAutoAgent",
                file
            )
        except Exception as e:
            return {
                "file": file,
                "error": str(e)
            }

tasks = [
    asyncio.create_task(safe_fetch(file))
    for file in repo_data
]

results = await asyncio.gather(*tasks)

In [17]:
len(repo_data)

121

In [18]:
len(results)

121

In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate
from langchain_core.prompts import HumanMessagePromptTemplate

template = """You are an advanced software architecture and repository intelligence agent.

Analyze the provided repository files and understand:
- repository structure
- architecture
- business purpose
- major features
- frontend/backend organization
- AI/automation workflows
- important business logic

Your goal is NOT to summarize every file individually.

Instead:
1. Infer what the project does
2. Detect frontend/backend/AI modules
3. Identify the most important files
4. Explain the architecture briefly
5. Detect core business logic
6. Detect AI workflows if present

Classify files into:
- HIGH importance
- MEDIUM importance
- LOW importance

HIGH importance files usually contain:
- APIs
- business logic
- prompts
- orchestration
- models
- services
- authentication
- workflows
- architecture

LOW importance files usually contain:
- tests
- migrations
- boilerplate
- generated files
- configs
- admin files

IMPORTANT:
Return ONLY the TOP 15 MOST IMPORTANT FILES.

Always include:
- requirements.txt if it exists
- package.json if it exists
- pyproject.toml if it exists
- Dockerfile if it exists

For each important file provide:
- file_name
- importance_score (1-10)
- reason

Keep responses concise.

DO NOT:
- generate markdown
- generate tables
- generate code blocks
- generate bullet lists
- generate explanations outside JSON
- add extra commentary

Return ONLY valid JSON in this exact format:

{{
  "files": [
    {{
      "file_name": "path/to/file",
      "importance_score": 8,
      "reason": "Contains core business logic for user authentication"
    }}
  ]
}}

Focus on concise repository intelligence."""
system_message = SystemMessagePromptTemplate.from_template(template)
human_message = HumanMessagePromptTemplate.from_template("{files}")
final_prompt = ChatPromptTemplate.from_messages([system_message, human_message])    

c:\Users\user\Desktop\RepoAnalyzer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
from langchain_groq.chat_models import ChatGroq


os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm=ChatGroq(model="openai/gpt-oss-120b", temperature=0.2)
# llm=model.with_structured_output(ResponseChatModel)
formatted_prompt = final_prompt.format(
    files=repo_data
)

result = llm.invoke(formatted_prompt)

In [21]:
import json
final_result=json.loads(result.content)

In [22]:
top_10_files = []
for item in final_result['files']:
    for key, value in item.items():
        if key == "file_name":
            top_10_files.append(value)

In [23]:
top_10_files

['sales/sales/settings.py',
 'sales/sales/urls.py',
 'sales/agent/prompts.py',
 'sales/agent/email_sender.py',
 'sales/agent/company_mail_agent.py',
 'sales/agent/graph.py',
 'sales/companies/models.py',
 'sales/campaigns/models.py',
 'sales/outreach/views.py',
 'sales/DataChunk/views.py',
 'frontend/src/api/client.ts',
 'frontend/src/components/AIAnalysisTab.tsx',
 'frontend/src/components/LoginPage.tsx',
 'requirements.txt',
 'frontend/package.json']

In [24]:
async def safe_fetch(file):
    async with SEM:
        try:
            return await get_file_content(
                "Akshat3422",
                "SalesAutoAgent",
                file
            )
        except Exception as e:
            return {
                "file": file,
                "error": str(e)
            }

tasks = [
    asyncio.create_task(safe_fetch(file))
    for file in top_10_files
]

result_data= await asyncio.gather(*tasks)

In [53]:
print(result_data[5])

import asyncio
import os
import re
import sys
import json
import logging
import socket
import requests
from typing import Dict, Any, List, TypedDict, Optional, Tuple
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin, urlunparse

from asgiref.sync import sync_to_async

# ──────────────────────────────────────────────────────────────────────────────
# Logger — UTF-8 forced on StreamHandler so Windows cp1252 never chokes on
# Unicode arrows (→) or other non-ASCII chars in log messages.
# ──────────────────────────────────────────────────────────────────────────────
logger = logging.getLogger("agent_pipeline")
logger.setLevel(logging.INFO)

_formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

# File handler — always UTF-8
_fh = logging.FileHandler(
    os.path.join(os.path.dirname(__file__), "agent_execution.log"),
    encoding="utf-8",
)
_fh.setFormatter(_formatter)
logger.addHandler(_fh)

# Console handler — wrap stdout in utf-8 w

In [26]:
from langgraph.graph import StateGraph,END
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
embedding=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

splitted_texts = []

for text in result_data:
    chunks = text_splitter.split_text(text)
    splitted_texts.extend(chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2476.36it/s]


In [27]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_texts(splitted_texts, embedding)

In [28]:
vector_store.save_local("faiss_index")

In [45]:
query = "Where is the core langgraph logiic stays ?"
similar_docs = vector_store.similarity_search(query, k=5)


In [46]:
final_content="".join(document.page_content for document in similar_docs)

In [47]:
rag_prompt = """
You are an advanced repository intelligence and software architecture assistant.

Use ONLY the provided repository context to answer the user's query.

Repository Context:
{final_content}

User Query:
{query}

Instructions:
- Answer based only on the retrieved repository context
- Focus on architecture, workflows, APIs, AI systems, and business logic
- If information is missing, say that the context does not provide enough information
- Keep answers technically accurate and concise
- Mention relevant file names when important
- Explain system interactions clearly
- Do not hallucinate nonexistent components
"""

In [48]:
result=llm.invoke(rag_prompt.format(final_content=final_content, query=query))

In [49]:
print(result.content)

**Answer**

The repository only shows a few fragments that *import* LangGraph components:

- `from langgraph.graph import StateGraph, START, END` (appears in the top‑level module that also defines `normalize_azure_base_url`, `call_llm`, and the logging setup)
- The same import is repeated later in the file that also pulls in Django models and other utilities.

However, the actual **core LangGraph workflow**—the place where a `StateGraph` is instantiated, nodes are added, edges are wired, and the graph is executed—is **not included in the provided snippets**. Those snippets contain only supporting code (environment handling, logging, model imports, and a return‑value helper), but they do not contain the graph definition or the functions that run it.

**So, based on the available context, the repository does not show where the core LangGraph logic lives.** If you need to locate it, look for a Python file in the project that:

1. Creates a `StateGraph` object (e.g., `graph = StateGraph(..